# Bangkok PM2.5 Forecasting — Step 3: Model Training

**Models**: ST-UNN (primary) + LSTM / GRU / MLP / Persistence (baselines)  
**Input**: Preprocessed data from `data/gold/model_ready/`  
**Output**: Trained models, evaluation metrics, SHAP analysis

### Models
| Model | Type | Description |
|-------|------|-------------|
| **Persistence** | Naive baseline | Predict PM2.5(t) = PM2.5(t-1) |
| **MLP** | Feed-forward | Flattened sequence → FC layers |
| **LSTM** | Recurrent | Standard LSTM encoder → FC head |
| **GRU** | Recurrent | GRU encoder → FC head |
| **ST-UNN** | Spatio-Temporal | Temporal encoder + Spatial attention + Feature fusion |

### Evaluation
- MAE (primary), RMSE (secondary)
- Walk-forward validation
- Seasonal breakdown
- Extreme pollution day accuracy (PM2.5 > 50 µg/m³)

In [1]:
import subprocess, sys

def _install_if_missing(packages: list[str]) -> None:
    """Install only packages that are not already importable."""
    missing = []
    import_map = {
        "scikit-learn": "sklearn",
        "pyarrow": "pyarrow",
        "geopandas": "geopandas",
        "ipykernel": "ipykernel",
    }
    for pkg in packages:
        mod = import_map.get(pkg, pkg.split("[")[0])
        try:
            __import__(mod)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    else:
        print("All packages already installed.")

_install_if_missing([
    "pandas", "pyarrow", "numpy", "requests",
    "matplotlib", "seaborn", "scikit-learn",
    "geopandas", "xarray", "shap",
])

# PyTorch ROCm — verify, never install generic CPU torch on top
try:
    import torch
    hip = getattr(torch.version, "hip", None)
    print(f"torch {torch.__version__} (ROCm: {hip or 'N/A'})")
except ImportError:
    print("ERROR: torch not found — install ROCm wheels manually:")
    print("  pip install <wheel> from https://repo.radeon.com/rocm/manylinux/rocm-rel-7.0.2/")

/home/a/rocm_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All packages already installed.
torch 2.10.0.dev20250926+rocm6.4 (ROCm: 6.4.43484-123eb5128)


In [2]:
from __future__ import annotations

import json
import logging
import os
import random
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

# ── Reproducibility: seed everything ────────────────────────────
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed: {SEED}  (Python, NumPy, PyTorch all seeded)")

# ── Device selection & diagnostics ──────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"\n{'─'*50}")
print(f"Device        : {DEVICE}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024 ** 3)
    print(f"GPU           : {props.name}")
    print(f"VRAM          : {vram_gb:.1f} GB")
    print(f"Compute Arch  : {props.major}.{props.minor}")
    print(f"SMs           : {props.multi_processor_count}")
    hip = getattr(torch.version, "hip", None)
    print(f"ROCm (HIP)    : {hip or 'N/A'}")
    print(f"PyTorch       : {torch.__version__}")
    print(f"CUDA avail    : {torch.cuda.is_available()}")
    print(f"Device count  : {torch.cuda.device_count()}")

    # Quick sanity — run a small tensor on GPU
    try:
        t = torch.randn(256, 256, device=DEVICE)
        _ = t @ t
        print(f"GPU matmul    : OK")
        del t
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"GPU matmul    : FAILED — {e}")
else:
    print("WARNING: No GPU detected — training will be slow on CPU.")
    print(f"PyTorch       : {torch.__version__}")

print(f"{'─'*50}")

Random seed: 42  (Python, NumPy, PyTorch all seeded)

──────────────────────────────────────────────────
Device        : cuda
GPU           : AMD Radeon RX 7800 XT
VRAM          : 16.0 GB
Compute Arch  : 11.0
SMs           : 30
ROCm (HIP)    : 6.4.43484-123eb5128
PyTorch       : 2.10.0.dev20250926+rocm6.4
CUDA avail    : True
Device count  : 1
GPU matmul    : OK
──────────────────────────────────────────────────


---
## 1. Configuration

In [3]:
@dataclass
class TrainConfig:
    """Training hyperparameters — single source of truth."""

    # Paths
    data_dir: Path = Path("data/gold/model_ready")
    output_dir: Path = Path("models")

    # Architecture
    hidden_size: int = 128
    num_layers: int = 2
    dropout: float = 0.2
    attention_heads: int = 4

    # Training
    epochs: int = 100
    batch_size: int = 64
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    patience: int = 15
    lr_scheduler_factor: float = 0.5
    lr_scheduler_patience: int = 7
    gradient_clip_norm: float = 1.0

    # Loss weights (MAE primary, RMSE secondary)
    mae_weight: float = 0.7
    rmse_weight: float = 0.3

    # Reproducibility (must match global SEED set in cell 2)
    seed: int = SEED


CFG = TrainConfig()
CFG.output_dir.mkdir(parents=True, exist_ok=True)


def setup_logging() -> logging.Logger:
    logger = logging.getLogger("training")
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        h = logging.StreamHandler()
        h.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S"))
        logger.addHandler(h)
    return logger


LOG = setup_logging()
LOG.info("Config: hidden=%d, layers=%d, lr=%.0e, epochs=%d, seed=%d",
         CFG.hidden_size, CFG.num_layers, CFG.learning_rate, CFG.epochs, CFG.seed)

19:14:16 [INFO] Config: hidden=128, layers=2, lr=1e-03, epochs=100, seed=42


---
## 2. Load Preprocessed Data

In [4]:
def load_manifest(data_dir: Path) -> dict:
    """Load pipeline manifest for feature/target metadata."""
    with open(data_dir / "pipeline_manifest.json") as f:
        return json.load(f)


MANIFEST = load_manifest(CFG.data_dir)
FEATURE_COLS = MANIFEST["feature_cols"]
TARGET_COL = MANIFEST["target_col"]
SEQ_LEN = MANIFEST["sequence_length"]
HORIZONS = MANIFEST["forecast_horizons"]
NUM_FEATURES = MANIFEST["num_features"]

print(f"Features     : {NUM_FEATURES}")
print(f"Sequence len : {SEQ_LEN} days")
print(f"Horizons     : {HORIZONS} days")
print(f"Target       : {TARGET_COL}")

Features     : 61
Sequence len : 30 days
Horizons     : [1, 3] days
Target       : pm2_5_ugm3


In [5]:
class PM25SequenceDataset(Dataset):
    """Sliding-window dataset: (seq_len × features) → PM2.5 at forecast horizons."""

    def __init__(
        self,
        df: pd.DataFrame,
        feature_cols: list[str],
        target_col: str,
        seq_len: int,
        horizons: list[int],
        station_col: str = "stationID",
    ):
        self.seq_len = seq_len
        self.horizons = horizons
        self.max_h = max(horizons)
        self.samples: list[tuple[np.ndarray, np.ndarray]] = []

        for _, sdf in df.groupby(station_col):
            sdf = sdf.sort_values("date").reset_index(drop=True)
            feats = sdf[feature_cols].values.astype(np.float32)
            tgts = sdf[target_col].values.astype(np.float32)

            for i in range(seq_len, len(sdf) - self.max_h + 1):
                x = feats[i - seq_len : i]
                y = np.array([tgts[i + h - 1] for h in horizons], dtype=np.float32)
                if np.isnan(x).any() or np.isnan(y).any():
                    continue
                self.samples.append((x, y))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        x, y = self.samples[idx]
        return torch.from_numpy(x), torch.from_numpy(y)


def load_split(name: str) -> pd.DataFrame:
    path = CFG.data_dir / f"{name}.parquet"
    df = pd.read_parquet(path)
    df["date"] = pd.to_datetime(df["date"])
    LOG.info("Loaded %s: %s rows", name, f"{len(df):,}")
    return df


df_train = load_split("train")
df_val = load_split("val")
df_test = load_split("test")

ds_train = PM25SequenceDataset(df_train, FEATURE_COLS, TARGET_COL, SEQ_LEN, HORIZONS)
ds_val = PM25SequenceDataset(df_val, FEATURE_COLS, TARGET_COL, SEQ_LEN, HORIZONS)
ds_test = PM25SequenceDataset(df_test, FEATURE_COLS, TARGET_COL, SEQ_LEN, HORIZONS)

LOG.info("Sequences — Train: %d, Val: %d, Test: %d", len(ds_train), len(ds_val), len(ds_test))

loader_train = DataLoader(ds_train, batch_size=CFG.batch_size, shuffle=False, drop_last=False)
loader_val = DataLoader(ds_val, batch_size=CFG.batch_size, shuffle=False)
loader_test = DataLoader(ds_test, batch_size=CFG.batch_size, shuffle=False)

if len(ds_train) > 0:
    x0, y0 = ds_train[0]
    print(f"Input shape : {x0.shape}")
    print(f"Target shape: {y0.shape}")
else:
    print("WARNING: 0 training sequences — PM2.5 data likely missing. Models will not train.")

19:14:16 [INFO] Loaded train: 156,435 rows
19:14:16 [INFO] Loaded val: 33,549 rows
19:14:16 [INFO] Loaded test: 29,067 rows
19:14:17 [INFO] Sequences — Train: 0, Val: 0, Test: 0


---
## 3. Loss Function

Combined MAE + RMSE loss as specified in ST-UNN design.

In [6]:
class CombinedLoss(nn.Module):
    """Weighted MAE + RMSE loss.

    MAE provides robust gradients; RMSE penalizes large errors more heavily.
    """

    def __init__(self, mae_weight: float = 0.7, rmse_weight: float = 0.3):
        super().__init__()
        self.mae_weight = mae_weight
        self.rmse_weight = rmse_weight

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        mae = torch.mean(torch.abs(pred - target))
        rmse = torch.sqrt(torch.mean((pred - target) ** 2) + 1e-8)
        return self.mae_weight * mae + self.rmse_weight * rmse


criterion = CombinedLoss(CFG.mae_weight, CFG.rmse_weight).to(DEVICE)

---
## 4. Model Definitions

### 4a. Persistence Baseline

In [7]:
class PersistenceModel:
    """Naive baseline: predict PM2.5(t+h) = PM2.5(t) for all horizons.

    Not a nn.Module — evaluated directly on DataLoader.
    """

    def __init__(self, pm25_lag1_idx: int, num_horizons: int):
        self.pm25_lag1_idx = pm25_lag1_idx
        self.num_horizons = num_horizons

    def predict(self, x: torch.Tensor) -> torch.Tensor:
        last_pm25 = x[:, -1, self.pm25_lag1_idx]
        return last_pm25.unsqueeze(1).expand(-1, self.num_horizons)


pm25_lag1_idx = FEATURE_COLS.index("pm2_5_ugm3_lag1") if "pm2_5_ugm3_lag1" in FEATURE_COLS else None
print(f"PM2.5 lag1 feature index: {pm25_lag1_idx}")

PM2.5 lag1 feature index: 22


### 4b. MLP Baseline

In [8]:
class MLPForecaster(nn.Module):
    """Simple feed-forward baseline: flatten sequence → hidden layers → forecast."""

    def __init__(self, seq_len: int, num_features: int, hidden_size: int, num_horizons: int, dropout: float = 0.2):
        super().__init__()
        input_dim = seq_len * num_features
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_horizons),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

### 4c. LSTM Forecaster

In [9]:
class LSTMForecaster(nn.Module):
    """LSTM-based sequence forecaster with FC regression head."""

    def __init__(self, num_features: int, hidden_size: int, num_layers: int, num_horizons: int, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=num_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_horizons),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

### 4d. GRU Forecaster

In [10]:
class GRUForecaster(nn.Module):
    """GRU-based sequence forecaster — lighter alternative to LSTM."""

    def __init__(self, num_features: int, hidden_size: int, num_layers: int, num_horizons: int, dropout: float = 0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=num_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_horizons),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.gru(x)
        return self.head(out[:, -1, :])

### 4e. ST-UNN (Spatio-Temporal Unified Neural Network)

Architecture per `st-unn.mdc`:  
1. **Temporal Encoder** — GRU captures sequential dependencies  
2. **Spatial Attention** — Multi-head attention weights feature importance (upwind, hotspot, etc.)  
3. **Feature Fusion** — Gated fusion of temporal and spatial representations  
4. **Regression Head** — FC layers output PM2.5 at each forecast horizon

In [11]:
class SpatialAttention(nn.Module):
    """Multi-head attention over the feature dimension to learn spatial/feature importance.

    Applied to the temporal encoder output at each timestep, allowing the model
    to dynamically weight features (e.g. upwind hotspots vs local weather).
    """

    def __init__(self, hidden_size: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Returns (attended_output, attention_weights)."""
        attn_out, attn_weights = self.attn(x, x, x)
        return self.norm(x + attn_out), attn_weights


class GatedFusion(nn.Module):
    """Gated fusion of temporal encoding and attention-weighted representation.

    gate = sigmoid(W_t * temporal + W_a * attended + b)
    output = gate * temporal + (1 - gate) * attended
    """

    def __init__(self, hidden_size: int):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Sigmoid(),
        )

    def forward(self, temporal: torch.Tensor, attended: torch.Tensor) -> torch.Tensor:
        combined = torch.cat([temporal, attended], dim=-1)
        g = self.gate(combined)
        return g * temporal + (1 - g) * attended


class STUNN(nn.Module):
    """Spatio-Temporal Unified Neural Network for PM2.5 forecasting.

    Pipeline:
        Input (batch, seq_len, features)
          → Input projection (features → hidden_size)
          → Temporal encoder (GRU, multi-layer)
          → Spatial attention (multi-head self-attention)
          → Gated fusion (temporal ⊕ attended)
          → Regression head → (batch, num_horizons)
    """

    def __init__(
        self,
        num_features: int,
        hidden_size: int,
        num_layers: int,
        num_horizons: int,
        attention_heads: int = 4,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.input_proj = nn.Linear(num_features, hidden_size)

        self.temporal_encoder = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )

        self.spatial_attention = SpatialAttention(hidden_size, attention_heads, dropout)
        self.fusion = GatedFusion(hidden_size)

        self.regression_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.GELU(),
            nn.Linear(hidden_size // 4, num_horizons),
        )

        self._last_attn_weights = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        proj = self.input_proj(x)

        temporal_out, _ = self.temporal_encoder(proj)

        attended, attn_w = self.spatial_attention(temporal_out)
        self._last_attn_weights = attn_w.detach()

        temporal_last = temporal_out[:, -1, :]
        attended_last = attended[:, -1, :]
        fused = self.fusion(temporal_last, attended_last)

        return self.regression_head(fused)

    @property
    def attention_weights(self) -> torch.Tensor | None:
        """Last forward pass attention weights for explainability."""
        return self._last_attn_weights

---
## 5. Training Engine

In [12]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    clip_norm: float,
) -> float:
    """Train for one epoch, return average loss."""
    model.train()
    total_loss = 0.0
    n_batches = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        pred = model(x_batch)
        loss = criterion(pred, y_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
) -> tuple[float, np.ndarray, np.ndarray]:
    """Evaluate model, return (loss, all_preds, all_targets)."""
    model.eval()
    total_loss = 0.0
    n_batches = 0
    preds_list, targets_list = [], []

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        pred = model(x_batch)
        loss = criterion(pred, y_batch)

        total_loss += loss.item()
        n_batches += 1
        preds_list.append(pred.cpu().numpy())
        targets_list.append(y_batch.cpu().numpy())

    all_preds = np.concatenate(preds_list) if preds_list else np.array([])
    all_targets = np.concatenate(targets_list) if targets_list else np.array([])
    return total_loss / max(n_batches, 1), all_preds, all_targets

In [13]:
def train_model(
    model: nn.Module,
    loader_train: DataLoader,
    loader_val: DataLoader,
    cfg: TrainConfig,
    model_name: str,
) -> dict:
    """Full training loop with early stopping, LR scheduling, and checkpointing."""
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=cfg.lr_scheduler_factor, patience=cfg.lr_scheduler_patience,
    )

    best_val_loss = float("inf")
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "lr": []}
    best_state = None

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    LOG.info("[%s] Parameters: %s, Device: %s", model_name, f"{n_params:,}", DEVICE)

    t_start = time.time()

    for epoch in range(1, cfg.epochs + 1):
        train_loss = train_one_epoch(model, loader_train, optimizer, criterion, cfg.gradient_clip_norm)
        val_loss, _, _ = evaluate(model, loader_val, criterion)

        current_lr = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(current_lr)

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        if epoch % 10 == 0 or epoch == 1:
            LOG.info("[%s] Epoch %3d/%d — train: %.4f, val: %.4f, lr: %.2e, patience: %d/%d",
                     model_name, epoch, cfg.epochs, train_loss, val_loss, current_lr, patience_counter, cfg.patience)

        if patience_counter >= cfg.patience:
            LOG.info("[%s] Early stopping at epoch %d", model_name, epoch)
            break

    elapsed = time.time() - t_start

    if best_state is not None:
        model.load_state_dict(best_state)
        ckpt_path = cfg.output_dir / f"{model_name}_best.pt"
        torch.save(best_state, ckpt_path)
        LOG.info("[%s] Best model saved → %s (val_loss=%.4f, %.1fs)", model_name, ckpt_path, best_val_loss, elapsed)

    return history

---
## 6. Metrics

In [14]:
def compute_metrics(preds: np.ndarray, targets: np.ndarray, horizons: list[int]) -> pd.DataFrame:
    """Compute MAE, RMSE, R² per forecast horizon."""
    rows = []
    for i, h in enumerate(horizons):
        p = preds[:, i]
        t = targets[:, i]
        mae = np.mean(np.abs(p - t))
        rmse = np.sqrt(np.mean((p - t) ** 2))
        ss_res = np.sum((t - p) ** 2)
        ss_tot = np.sum((t - np.mean(t)) ** 2)
        r2 = 1 - ss_res / (ss_tot + 1e-8)
        rows.append({"horizon_days": h, "MAE": mae, "RMSE": rmse, "R2": r2})
    return pd.DataFrame(rows)


def compute_seasonal_metrics(preds: np.ndarray, targets: np.ndarray, dates: pd.Series, horizons: list[int]) -> pd.DataFrame:
    """Breakdown metrics by Thai pollution seasons."""
    months = dates.dt.month.values[:len(preds)]

    seasons = np.where(
        np.isin(months, [2, 3, 4]), "Burning (Feb-Apr)",
        np.where(np.isin(months, [5, 6, 7, 8, 9]), "Monsoon (May-Sep)", "Cool (Oct-Jan)")
    )

    rows = []
    for season in ["Burning (Feb-Apr)", "Monsoon (May-Sep)", "Cool (Oct-Jan)"]:
        mask = seasons == season
        if mask.sum() == 0:
            continue
        for i, h in enumerate(horizons):
            p = preds[mask, i]
            t = targets[mask, i]
            rows.append({
                "season": season,
                "horizon_days": h,
                "MAE": np.mean(np.abs(p - t)),
                "RMSE": np.sqrt(np.mean((p - t) ** 2)),
                "n_samples": int(mask.sum()),
            })
    return pd.DataFrame(rows)


def compute_extreme_accuracy(preds: np.ndarray, targets: np.ndarray, threshold: float = 50.0) -> dict:
    """Accuracy metrics for extreme pollution days (PM2.5 > threshold)."""
    actual_extreme = targets[:, 0] > threshold
    pred_extreme = preds[:, 0] > threshold
    n_extreme = actual_extreme.sum()
    if n_extreme == 0:
        return {"n_extreme_days": 0, "detection_rate": 0, "false_alarm_rate": 0}

    tp = (actual_extreme & pred_extreme).sum()
    fp = (~actual_extreme & pred_extreme).sum()

    return {
        "n_extreme_days": int(n_extreme),
        "detection_rate": tp / n_extreme,
        "false_alarm_rate": fp / max((~actual_extreme).sum(), 1),
    }

---
## 7. Train All Models

In [15]:
NUM_HORIZONS = len(HORIZONS)


def build_all_models() -> dict[str, nn.Module]:
    """Instantiate all model architectures."""
    models = {
        "MLP": MLPForecaster(SEQ_LEN, NUM_FEATURES, CFG.hidden_size * 2, NUM_HORIZONS, CFG.dropout),
        "LSTM": LSTMForecaster(NUM_FEATURES, CFG.hidden_size, CFG.num_layers, NUM_HORIZONS, CFG.dropout),
        "GRU": GRUForecaster(NUM_FEATURES, CFG.hidden_size, CFG.num_layers, NUM_HORIZONS, CFG.dropout),
        "ST-UNN": STUNN(NUM_FEATURES, CFG.hidden_size, CFG.num_layers, NUM_HORIZONS, CFG.attention_heads, CFG.dropout),
    }

    for name, m in models.items():
        n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
        print(f"  {name:8s}: {n_params:>10,} parameters")

    return models


if len(ds_train) == 0:
    print("Cannot train — no valid sequences. PM2.5 data is required.")
    print("Fix AQ ingestion in Step 1, re-run Step 2, then come back here.")
    models = build_all_models()
    histories = {}
else:
    models = build_all_models()
    histories = {}
    for name, model in models.items():
        print(f"\n{'='*50}")
        print(f"  Training: {name}")
        print(f"{'='*50}")
        histories[name] = train_model(model, loader_train, loader_val, CFG, name)

Cannot train — no valid sequences. PM2.5 data is required.
Fix AQ ingestion in Step 1, re-run Step 2, then come back here.
  MLP     :    501,890 parameters
  LSTM    :    238,274 parameters
  GRU     :    180,802 parameters
  ST-UNN  :    315,682 parameters


---
## 8. Evaluate on Test Set

In [16]:
results = {}

if len(ds_test) > 0:
    # Persistence baseline
    if pm25_lag1_idx is not None:
        persist = PersistenceModel(pm25_lag1_idx, NUM_HORIZONS)
        p_preds, p_targets = [], []
        for x_batch, y_batch in loader_test:
            p_preds.append(persist.predict(x_batch).numpy())
            p_targets.append(y_batch.numpy())
        results["Persistence"] = {
            "preds": np.concatenate(p_preds),
            "targets": np.concatenate(p_targets),
        }

    # Neural models
    for name, model in models.items():
        model = model.to(DEVICE)
        _, preds, targets = evaluate(model, loader_test, criterion)
        results[name] = {"preds": preds, "targets": targets}

    # Metrics comparison
    all_metrics = []
    for name, res in results.items():
        df_m = compute_metrics(res["preds"], res["targets"], HORIZONS)
        df_m["model"] = name
        all_metrics.append(df_m)

    metrics_df = pd.concat(all_metrics, ignore_index=True)
    print("\n" + "=" * 60)
    print("  TEST SET METRICS")
    print("=" * 60)
    print(metrics_df.pivot_table(index="model", columns="horizon_days", values=["MAE", "RMSE", "R2"]).round(3).to_string())

    metrics_df.to_csv(CFG.output_dir / "test_metrics.csv", index=False)
else:
    print("No test sequences available — skipping evaluation.")

No test sequences available — skipping evaluation.


---
## 9. Seasonal & Extreme Day Analysis

In [17]:
if results:
    test_dates = df_test.sort_values(["stationID", "date"])["date"]

    print("\n--- Seasonal Breakdown (ST-UNN) ---")
    if "ST-UNN" in results:
        seasonal = compute_seasonal_metrics(results["ST-UNN"]["preds"], results["ST-UNN"]["targets"], test_dates, HORIZONS)
        print(seasonal.to_string(index=False))

    print("\n--- Extreme Pollution Day Detection ---")
    for name, res in results.items():
        ext = compute_extreme_accuracy(res["preds"], res["targets"])
        print(f"  {name:12s}: {ext}")
else:
    print("No results to analyze.")

No results to analyze.


---
## 10. Training Curves & Comparison Plots

In [18]:
if histories:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    for name, h in histories.items():
        ax.plot(h["train_loss"], label=f"{name} (train)", alpha=0.7)
        ax.plot(h["val_loss"], label=f"{name} (val)", linestyle="--", alpha=0.7)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training & Validation Loss")
    ax.legend(fontsize=8)

    ax = axes[1]
    if results:
        model_names = list(results.keys())
        mae_24h = [compute_metrics(results[n]["preds"], results[n]["targets"], HORIZONS).iloc[0]["MAE"] for n in model_names]
        colors = sns.color_palette("viridis", len(model_names))
        bars = ax.bar(model_names, mae_24h, color=colors)
        ax.set_ylabel("MAE (µg/m³)")
        ax.set_title("24h Forecast — MAE Comparison")
        for bar, v in zip(bars, mae_24h):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{v:.2f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.savefig(CFG.output_dir / "training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No training histories — skipping plots.")

No training histories — skipping plots.


In [19]:
if results and "ST-UNN" in results:
    fig, axes = plt.subplots(1, len(HORIZONS), figsize=(7 * len(HORIZONS), 6))
    if len(HORIZONS) == 1:
        axes = [axes]

    for i, (ax, h) in enumerate(zip(axes, HORIZONS)):
        p = results["ST-UNN"]["preds"][:, i]
        t = results["ST-UNN"]["targets"][:, i]
        ax.scatter(t, p, alpha=0.15, s=4, color="tab:blue")
        lims = [min(t.min(), p.min()), max(t.max(), p.max())]
        ax.plot(lims, lims, "r--", linewidth=1, label="Perfect")
        ax.set_xlabel(f"Actual PM2.5 (+{h}d)")
        ax.set_ylabel(f"Predicted PM2.5 (+{h}d)")
        ax.set_title(f"ST-UNN — {h}-Day Forecast")
        ax.legend()

    plt.tight_layout()
    plt.savefig(CFG.output_dir / "stunn_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()

---
## 11. SHAP Feature Importance (ST-UNN)

In [20]:
if len(ds_test) > 0 and "ST-UNN" in models:
    import shap

    stunn_model = models["ST-UNN"].to(DEVICE).eval()

    shap_sample_size = min(200, len(ds_test))
    bg_size = min(50, len(ds_train))

    sample_x = torch.stack([ds_test[i][0] for i in range(shap_sample_size)]).to(DEVICE)
    bg_x = torch.stack([ds_train[i][0] for i in range(bg_size)]).to(DEVICE)

    def model_predict_24h(x: torch.Tensor) -> torch.Tensor:
        """Wrapper returning only the 24h horizon for SHAP."""
        with torch.no_grad():
            return stunn_model(x)[:, 0:1]

    explainer = shap.GradientExplainer(model_predict_24h, bg_x)
    shap_values = explainer.shap_values(sample_x)

    # Average absolute SHAP over time steps → per-feature importance
    if isinstance(shap_values, list):
        sv = shap_values[0]
    else:
        sv = shap_values
    mean_abs_shap = np.mean(np.abs(sv), axis=(0, 1))  # (features,)

    feat_importance = pd.DataFrame({
        "feature": FEATURE_COLS,
        "mean_abs_shap": mean_abs_shap,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    print("\n--- Top 15 Features (SHAP) ---")
    print(feat_importance.head(15).to_string(index=False))

    feat_importance.to_csv(CFG.output_dir / "feature_importance_shap.csv", index=False)

    fig, ax = plt.subplots(figsize=(10, 8))
    top20 = feat_importance.head(20)
    ax.barh(top20["feature"][::-1], top20["mean_abs_shap"][::-1], color="steelblue")
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title("ST-UNN Feature Importance (24h Forecast)")
    plt.tight_layout()
    plt.savefig(CFG.output_dir / "shap_importance.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("SHAP analysis requires test sequences + trained ST-UNN. Skipping.")

SHAP analysis requires test sequences + trained ST-UNN. Skipping.


---
## 12. Export Best Model for Deployment

In [21]:
if len(ds_train) > 0 and "ST-UNN" in models:
    export_path = CFG.output_dir / "stunn_deployment.pt"

    deployment_bundle = {
        "model_state_dict": models["ST-UNN"].cpu().state_dict(),
        "config": {
            "num_features": NUM_FEATURES,
            "hidden_size": CFG.hidden_size,
            "num_layers": CFG.num_layers,
            "num_horizons": NUM_HORIZONS,
            "attention_heads": CFG.attention_heads,
            "dropout": CFG.dropout,
            "seq_len": SEQ_LEN,
        },
        "manifest": MANIFEST,
        "feature_cols": FEATURE_COLS,
        "horizons": HORIZONS,
    }

    torch.save(deployment_bundle, export_path)
    LOG.info("Deployment bundle saved → %s (%.1f MB)", export_path, export_path.stat().st_size / 1e6)
else:
    print("No trained model to export.")

No trained model to export.


---
## 13. Forecasting with Trained Models

Generate predictions from all trained models on the test set, visualize forecast quality,
and provide a reusable `forecast()` function for deployment.

In [ ]:
@torch.no_grad()
def forecast_all_models(
    models_dict: dict[str, nn.Module],
    loader: DataLoader,
    horizons: list[int],
    persist_model: PersistenceModel | None = None,
) -> dict[str, dict[str, np.ndarray]]:
    """Run inference on all models, return {name: {preds, targets}} per model."""
    all_results: dict[str, dict[str, np.ndarray]] = {}

    # Persistence baseline
    if persist_model is not None:
        p_preds, p_targets = [], []
        for x_b, y_b in loader:
            p_preds.append(persist_model.predict(x_b).numpy())
            p_targets.append(y_b.numpy())
        all_results["Persistence"] = {
            "preds": np.concatenate(p_preds),
            "targets": np.concatenate(p_targets),
        }

    # Neural models
    for name, model in models_dict.items():
        model = model.to(DEVICE).eval()
        m_preds, m_targets = [], []
        for x_b, y_b in loader:
            pred = model(x_b.to(DEVICE)).cpu().numpy()
            m_preds.append(pred)
            m_targets.append(y_b.numpy())
        all_results[name] = {
            "preds": np.concatenate(m_preds),
            "targets": np.concatenate(m_targets),
        }

    return all_results


if len(ds_test) > 0:
    forecast_results = forecast_all_models(
        models, loader_test, HORIZONS,
        persist_model=PersistenceModel(pm25_lag1_idx, NUM_HORIZONS) if pm25_lag1_idx is not None else None,
    )
    LOG.info("Forecast generated for %d models, %d test samples", len(forecast_results), len(ds_test))
else:
    forecast_results = {}
    print("No test sequences — forecasting requires PM2.5 data.")

### 13a. Forecast Time Series — Actual vs Predicted

In [ ]:
if forecast_results:
    best_model_name = "ST-UNN" if "ST-UNN" in forecast_results else list(forecast_results.keys())[-1]

    for h_idx, h_days in enumerate(HORIZONS):
        fig, axes = plt.subplots(2, 1, figsize=(18, 10), gridspec_kw={"height_ratios": [3, 1]})

        actual = forecast_results[best_model_name]["targets"][:, h_idx]
        n_plot = min(500, len(actual))
        x_axis = np.arange(n_plot)

        # Top: time-series overlay
        ax = axes[0]
        ax.plot(x_axis, actual[:n_plot], color="black", linewidth=1, alpha=0.8, label="Actual")

        model_colors = {"Persistence": "#9E9E9E", "MLP": "#FF9800", "LSTM": "#2196F3", "GRU": "#4CAF50", "ST-UNN": "#F44336"}
        for name, res in forecast_results.items():
            pred = res["preds"][:n_plot, h_idx]
            color = model_colors.get(name, "gray")
            lw = 1.5 if name == best_model_name else 0.8
            alpha = 0.9 if name == best_model_name else 0.5
            ax.plot(x_axis, pred, color=color, linewidth=lw, alpha=alpha, label=name)

        ax.axhline(50, color="red", linestyle=":", linewidth=0.8, alpha=0.4, label="Unhealthy (50 µg/m³)")
        ax.set_ylabel("PM2.5 (µg/m³)")
        ax.set_title(f"Forecast vs Actual — +{h_days} Day Horizon (first {n_plot} samples)")
        ax.legend(fontsize=8, ncol=3, loc="upper right")

        # Bottom: residuals for best model
        ax = axes[1]
        residuals = forecast_results[best_model_name]["preds"][:n_plot, h_idx] - actual[:n_plot]
        ax.bar(x_axis, residuals, width=1, color=np.where(residuals > 0, "#F44336", "#2196F3"), alpha=0.6)
        ax.axhline(0, color="black", linewidth=0.5)
        ax.set_ylabel("Residual")
        ax.set_xlabel("Sample index")
        ax.set_title(f"{best_model_name} Residuals — mean={residuals.mean():.2f}, std={residuals.std():.2f}")

        plt.tight_layout()
        plt.savefig(CFG.output_dir / f"forecast_timeseries_{h_days}d.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("No forecast results to plot.")

### 13b. Error Distribution per Model

In [ ]:
if forecast_results:
    n_models = len(forecast_results)

    for h_idx, h_days in enumerate(HORIZONS):
        fig, axes = plt.subplots(1, n_models, figsize=(4.5 * n_models, 5), sharey=True)
        if n_models == 1:
            axes = [axes]

        for i, (name, res) in enumerate(forecast_results.items()):
            ax = axes[i]
            errors = res["preds"][:, h_idx] - res["targets"][:, h_idx]
            color = model_colors.get(name, "gray")

            ax.hist(errors, bins=60, color=color, alpha=0.7, edgecolor="white", density=True)
            ax.axvline(0, color="black", linewidth=0.8)
            ax.axvline(errors.mean(), color="red", linestyle="--", linewidth=1,
                       label=f"bias={errors.mean():.2f}")
            ax.set_xlabel("Error (pred - actual)")
            ax.set_title(f"{name}\nMAE={np.abs(errors).mean():.2f}")
            ax.legend(fontsize=8)

        axes[0].set_ylabel("Density")
        fig.suptitle(f"Forecast Error Distribution — +{h_days} Day", fontsize=13, y=1.02)
        plt.tight_layout()
        plt.savefig(CFG.output_dir / f"forecast_error_dist_{h_days}d.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("No forecast results to plot.")

### 13c. Rolling Forecast Accuracy (MAE over time)

In [ ]:
if forecast_results:
    ROLLING_WIN = 50

    for h_idx, h_days in enumerate(HORIZONS):
        fig, ax = plt.subplots(figsize=(16, 5))

        for name, res in forecast_results.items():
            abs_err = np.abs(res["preds"][:, h_idx] - res["targets"][:, h_idx])
            rolling_mae = pd.Series(abs_err).rolling(ROLLING_WIN, min_periods=1).mean()
            color = model_colors.get(name, "gray")
            lw = 2 if name == best_model_name else 1
            ax.plot(rolling_mae.values, color=color, linewidth=lw, alpha=0.8, label=name)

        ax.set_xlabel("Sample index")
        ax.set_ylabel(f"Rolling MAE (window={ROLLING_WIN})")
        ax.set_title(f"Rolling Forecast Accuracy — +{h_days} Day Horizon")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(CFG.output_dir / f"forecast_rolling_mae_{h_days}d.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("No forecast results to plot.")

### 13d. Forecast by PM2.5 Intensity Band

In [ ]:
if forecast_results:
    AQI_BANDS = [
        (0, 25, "Good (0-25)", "#4CAF50"),
        (25, 50, "Moderate (25-50)", "#FF9800"),
        (50, 100, "Unhealthy-Sensitive (50-100)", "#FF5722"),
        (100, float("inf"), "Unhealthy (100+)", "#F44336"),
    ]

    h_idx = 0
    h_days = HORIZONS[h_idx]
    actual = forecast_results[best_model_name]["targets"][:, h_idx]

    rows = []
    for name, res in forecast_results.items():
        pred = res["preds"][:, h_idx]
        for lo, hi, label, _ in AQI_BANDS:
            mask = (actual >= lo) & (actual < hi)
            if mask.sum() == 0:
                continue
            mae = np.abs(pred[mask] - actual[mask]).mean()
            rows.append({"model": name, "band": label, "MAE": mae, "n_samples": int(mask.sum())})

    band_df = pd.DataFrame(rows)

    if len(band_df) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        band_pivot = band_df.pivot(index="band", columns="model", values="MAE")
        band_pivot = band_pivot.reindex([b[2] for b in AQI_BANDS if b[2] in band_pivot.index])
        band_pivot.plot(kind="bar", ax=ax, width=0.75)
        ax.set_ylabel("MAE (µg/m³)")
        ax.set_title(f"Forecast MAE by PM2.5 Intensity Band — +{h_days} Day")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right")
        ax.legend(fontsize=8, title="Model")
        ax.grid(axis="y", alpha=0.3)

        plt.tight_layout()
        plt.savefig(CFG.output_dir / "forecast_by_aqi_band.png", dpi=150, bbox_inches="tight")
        plt.show()

        print(band_df.to_string(index=False))
else:
    print("No forecast results to plot.")

### 13e. Multi-Horizon Comparison (24h vs 72h)

In [ ]:
if forecast_results and len(HORIZONS) > 1:
    fig, axes = plt.subplots(1, len(HORIZONS), figsize=(8 * len(HORIZONS), 6))
    if len(HORIZONS) == 1:
        axes = [axes]

    for h_idx, (ax, h_days) in enumerate(zip(axes, HORIZONS)):
        for name, res in forecast_results.items():
            pred = res["preds"][:, h_idx]
            actual = res["targets"][:, h_idx]
            color = model_colors.get(name, "gray")
            alpha = 0.25 if name == "Persistence" else 0.15
            s = 6 if name == best_model_name else 3
            ax.scatter(actual, pred, alpha=alpha, s=s, color=color, label=name)

        lims = [
            min(ax.get_xlim()[0], ax.get_ylim()[0]),
            max(ax.get_xlim()[1], ax.get_ylim()[1]),
        ]
        ax.plot(lims, lims, "k--", linewidth=1, alpha=0.5, label="Perfect")
        ax.set_xlabel(f"Actual PM2.5 (+{h_days}d)")
        ax.set_ylabel(f"Predicted PM2.5 (+{h_days}d)")
        ax.set_title(f"+{h_days} Day Horizon — All Models")
        ax.legend(fontsize=7, markerscale=3)
        ax.set_aspect("equal")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(CFG.output_dir / "forecast_multi_horizon_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()
elif forecast_results:
    print("Single horizon — scatter already shown in section 10.")
else:
    print("No forecast results to plot.")

### 13f. Reusable `forecast()` Function for Deployment

Load a saved model bundle and run inference on new data.

In [ ]:
def load_deployment_model(bundle_path: str | Path, device: str = "cuda") -> tuple[STUNN, dict]:
    """Load a saved deployment bundle and return (model, metadata).

    Usage:
        model, meta = load_deployment_model("models/stunn_deployment.pt")
        pred = forecast(model, input_tensor, meta)
    """
    bundle = torch.load(bundle_path, map_location=device, weights_only=False)
    cfg = bundle["config"]

    model = STUNN(
        num_features=cfg["num_features"],
        hidden_size=cfg["hidden_size"],
        num_layers=cfg["num_layers"],
        num_horizons=cfg["num_horizons"],
        attention_heads=cfg["attention_heads"],
        dropout=cfg["dropout"],
    )
    model.load_state_dict(bundle["model_state_dict"])
    model = model.to(device).eval()

    return model, bundle


@torch.no_grad()
def forecast(
    model: nn.Module,
    input_sequence: np.ndarray | torch.Tensor,
    feature_cols: list[str] | None = None,
    horizons: list[int] | None = None,
) -> pd.DataFrame:
    """Run a single forecast from a (seq_len, num_features) input array.

    Args:
        model: Trained model in eval mode.
        input_sequence: Shape (seq_len, num_features) — the last N days of features.
        feature_cols: Column names (for logging).
        horizons: Forecast horizons in days.

    Returns:
        DataFrame with columns [horizon_days, pm25_forecast].
    """
    if isinstance(input_sequence, np.ndarray):
        input_sequence = torch.from_numpy(input_sequence).float()

    if input_sequence.dim() == 2:
        input_sequence = input_sequence.unsqueeze(0)

    device = next(model.parameters()).device
    pred = model(input_sequence.to(device)).cpu().numpy().squeeze()

    if horizons is None:
        horizons = list(range(1, len(pred) + 1))

    return pd.DataFrame({
        "horizon_days": horizons,
        "pm25_forecast": pred,
    })


# Demo: run forecast on the last test sample
if len(ds_test) > 0 and "ST-UNN" in models:
    demo_x, demo_y = ds_test[-1]
    demo_result = forecast(models["ST-UNN"].to(DEVICE), demo_x, FEATURE_COLS, HORIZONS)
    print("--- Demo Forecast (last test sample) ---")
    print(demo_result.to_string(index=False))
    print(f"\nActual values: {demo_y.numpy()}")
else:
    print("Demo forecast skipped — no trained model or test data.")

### 13g. Forecast Summary Table

In [ ]:
if forecast_results:
    summary_rows = []
    for name, res in forecast_results.items():
        for h_idx, h_days in enumerate(HORIZONS):
            p = res["preds"][:, h_idx]
            t = res["targets"][:, h_idx]
            mae = np.mean(np.abs(p - t))
            rmse = np.sqrt(np.mean((p - t) ** 2))
            bias = np.mean(p - t)
            ss_res = np.sum((t - p) ** 2)
            ss_tot = np.sum((t - np.mean(t)) ** 2)
            r2 = 1 - ss_res / (ss_tot + 1e-8)

            extreme_mask = t > 50
            extreme_mae = np.mean(np.abs(p[extreme_mask] - t[extreme_mask])) if extreme_mask.sum() > 0 else float("nan")

            summary_rows.append({
                "Model": name,
                "Horizon": f"+{h_days}d",
                "MAE": round(mae, 2),
                "RMSE": round(rmse, 2),
                "Bias": round(bias, 2),
                "R²": round(r2, 3),
                "Extreme MAE (>50)": round(extreme_mae, 2) if not np.isnan(extreme_mae) else "N/A",
                "N": len(t),
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(CFG.output_dir / "forecast_summary.csv", index=False)

    print("=" * 80)
    print("  FORECAST PERFORMANCE SUMMARY")
    print("=" * 80)
    print(summary_df.to_string(index=False))
    print(f"\nSaved → {CFG.output_dir / 'forecast_summary.csv'}")
else:
    print("No forecast results — PM2.5 data needed.")

---
## 14. Final Summary

In [22]:
print("\n" + "=" * 70)
print("  MODEL TRAINING — FINAL SUMMARY")
print("=" * 70)

print(f"\n--- Data ---")
print(f"  Train sequences: {len(ds_train):,}")
print(f"  Val sequences  : {len(ds_val):,}")
print(f"  Test sequences : {len(ds_test):,}")

if results:
    print(f"\n--- Best Test MAE (24h) ---")
    for name, res in results.items():
        m = compute_metrics(res["preds"], res["targets"], HORIZONS)
        print(f"  {name:12s}: MAE={m.iloc[0]['MAE']:.2f}, RMSE={m.iloc[0]['RMSE']:.2f}, R²={m.iloc[0]['R2']:.3f}")

print(f"\n--- Outputs ---")
for f in sorted(CFG.output_dir.iterdir()):
    print(f"  {f.name:40s} ({f.stat().st_size / 1e6:.1f} MB)")

if len(ds_train) == 0:
    print("\n*** WARNING: No training was performed — PM2.5 data is missing! ***")
    print("    Fix AQ ingestion → re-run preprocessing → re-run this notebook.")

print("\n" + "=" * 70)


  MODEL TRAINING — FINAL SUMMARY

--- Data ---
  Train sequences: 0
  Val sequences  : 0
  Test sequences : 0

--- Outputs ---

*** WARNING: No training was performed — PM2.5 data is missing! ***
    Fix AQ ingestion → re-run preprocessing → re-run this notebook.

